# Dask Performance: Lazy Evaluation and Speedup

This notebook covers two key Dask concepts: **lazy evaluation** (task graphs, when to call `.compute()`) and **parallel speedup** (benchmarking Pandas vs Dask on a 2D histogram problem).

In [ ]:
import pathlib
import time
import dask.dataframe as dd
import dask.array as da
from dask.distributed import Client
import psutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
%matplotlib inline

print(f"Available RAM: {psutil.virtual_memory().available / (1024 ** 3):.1f} GB")

## Part 1: Lazy Evaluation and Task Graphs

Dask operations build a computation graph instead of executing immediately. This enables optimization before execution.

In [ ]:
# Load the weather HDF5 we created in the previous notebook
data_path = pathlib.Path("meteo") / "meteo_dask.h5"
assert data_path.exists(), f"Run notebook 02 first to create {data_path}"
df = dd.read_hdf(data_path, key="df")

In [ ]:
# .head() is eager — it triggers computation of the first few rows
df.head()

In [ ]:
# This is LAZY — returns a Dask Scalar, not a number
lazy_result = df[df.HumOut > 0].TempOut.mean()
lazy_result

In [ ]:
# .compute() triggers execution of the entire graph
lazy_result.compute()

### Task Graph Visualization

Uncomment to see the computation graph. This shows how Dask breaks the operation into parallel tasks across partitions.

In [ ]:
# df[df.HumOut > 0].TempOut.mean().visualize(engine="graphviz")

### GroupBy: Distributed Aggregation

GroupBy requires data shuffling between partitions — a more complex operation than simple filtering.

In [ ]:
%%time
df.TempOut.groupby(df.HumOut).mean().compute()

## Part 2: Pandas vs Dask Benchmark

We compare single-threaded Pandas with distributed Dask on a CPU-intensive task: 2D histogram of 100M simulated physics measurements.

In [ ]:
n_samples = 100_000_000
distance_mean_m = 100
distance_sigma_m = 10
time_mean_s = 10
time_sigma_s = 0.1

cmap = mcolors.LinearSegmentedColormap.from_list("custom", [(0,0,0,0), (0,1,0,1), (1,0,0,1)], N=128)

### Pandas Approach (single-threaded)

In [ ]:
%%time
df_pandas = pd.DataFrame({
    "distance_m": np.random.normal(loc=distance_mean_m, scale=distance_sigma_m, size=n_samples),
    "time_s": np.random.normal(loc=time_mean_s, scale=time_sigma_s, size=n_samples),
})

In [ ]:
%%time
hist_pandas, xedges, yedges = np.histogram2d(df_pandas.distance_m, df_pandas.time_s, bins=500)

In [ ]:
norm = mcolors.LogNorm(vmin=1, vmax=hist_pandas.max())
plt.imshow(hist_pandas.T, origin="lower", aspect="auto",
           extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]],
           cmap=cmap, norm=norm)
plt.xlabel("Distance (m)")
plt.ylabel("Time (s)")
plt.title("Pandas: np.histogram2d")
plt.colorbar(label="Counts")
plt.grid()

### Dask Approach (distributed)

Start a local Dask cluster. The dashboard (default port 8787) shows live task execution and memory usage.

In [ ]:
# Use your assigned port from 00_setup (change if needed)
DASHBOARD_PORT = 8788  # adjust to your participant number
client = Client(dashboard_address=f":{DASHBOARD_PORT}")
client

In [ ]:
# Create lazy distributed arrays
distance_dask = da.random.normal(loc=distance_mean_m, scale=distance_sigma_m, size=n_samples, chunks="auto")
time_dask = da.random.normal(loc=time_mean_s, scale=time_sigma_s, size=n_samples, chunks="auto")

In [ ]:
# Pre-compute range for consistent binning across workers
d_min, d_max, t_min, t_max = da.compute(
    distance_dask.min(), distance_dask.max(),
    time_dask.min(), time_dask.max()
)

In [ ]:
%%time
hist_dask, xedges_d, yedges_d = da.histogram2d(
    distance_dask, time_dask, bins=(500, 500),
    range=[[d_min, d_max], [t_min, t_max]]
)

In [ ]:
%%time
hist_dask, xedges_d, yedges_d = da.compute(hist_dask, xedges_d, yedges_d)

In [ ]:
norm = mcolors.LogNorm(vmin=1, vmax=hist_dask.max())
plt.imshow(hist_dask.T, origin="lower", aspect="auto",
           extent=[xedges_d[0], xedges_d[-1], yedges_d[0], yedges_d[-1]],
           cmap=cmap, norm=norm)
plt.xlabel("Distance (m)")
plt.ylabel("Time (s)")
plt.title("Dask: da.histogram2d")
plt.colorbar(label="Counts")
plt.grid()

## Exercise: Find the Crossover Point

At what dataset size does Dask become faster than Pandas for the 2D histogram? Test multiple sizes and plot the results.

In [ ]:
# Exercise: benchmark Pandas vs Dask at different dataset sizes

sample_sizes = [10_000, 100_000, 1_000_000, 10_000_000, 100_000_000]
pandas_times = []
dask_times = []

for n in sample_sizes:
    # YOUR CODE HERE: time the Pandas approach (np.histogram2d)
    pass

    # YOUR CODE HERE: time the Dask approach (da.histogram2d + .compute())
    pass

# YOUR CODE HERE: plot sample_sizes vs times for both
# plt.loglog(sample_sizes, pandas_times, "o-", label="Pandas")
# plt.loglog(sample_sizes, dask_times, "s-", label="Dask")
# plt.xlabel("Dataset size")
# plt.ylabel("Wall time (s)")
# plt.legend()
# plt.grid(True)

In [ ]:
client.close()